# Ejercicio 9: Uso de la API de Google Gemini


## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

Se instaló el SDK nuevo de Google (`google-genai`).

In [ ]:
!pip install google-genai

In [2]:
from google import genai
import numpy as np

Se creó el cliente con la API key obtenida de Google AI Studio.

In [ ]:
client = genai.Client(api_key="API")

Se probó que la conexión funcione mandando un prompt sencillo a `gemini-2.5-flash`.

In [4]:
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Explica en 2 oraciones qué es la recuperación de información."
)
print(response.text)

La recuperación de información es la disciplina informática que se encarga de buscar, organizar y localizar datos relevantes dentro de grandes colecciones de documentos no estructurados. Su objetivo principal es responder de manera rápida y precisa a las consultas de los usuarios, facilitando el acceso a los recursos que mejor satisfagan su necesidad de conocimiento.


## 2. Retrieval

### 2.1 Cargar el corpus de 20 News Groups

Se cargó el dataset desde sklearn. Se removieron headers y footers para quedarse solo con el contenido de las noticias. Se tomaron 500 documentos para que el proceso de embeddings no tarde demasiado.

In [5]:
from sklearn.datasets import fetch_20newsgroups

newsgroups = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))

N_DOCS = 500
docs = newsgroups.data[:N_DOCS]
labels = newsgroups.target[:N_DOCS]
nombres_categorias = newsgroups.target_names

print(f"Documentos cargados: {len(docs)}")
print(f"Categorías disponibles: {len(nombres_categorias)}")
print(f"\nEjemplo (primeros 300 chars):")
print(docs[0][:300])

Documentos cargados: 500
Categorías disponibles: 20

Ejemplo (primeros 300 chars):
I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I k


### 2.2 Transformar a embeddings

Se utilizó `all-MiniLM-L6-v2` de sentence-transformers para generar los embeddings de forma local. Este modelo genera vectores de 384 dimensiones y es liviano (~80MB).

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Modelo cargado. Dimensión de embeddings: {encoder.get_embedding_dimension()}")

Se generaron los embeddings de todo el corpus. Se normalizaron los vectores para que el producto punto equivalga a la similitud coseno.

In [ ]:
embeddings_corpus = encoder.encode(docs, show_progress_bar=True, normalize_embeddings=True)

print(f"Shape de la matriz: {embeddings_corpus.shape}")

### 2.3 Crear y ejecutar query

Se codificó la query con el mismo modelo y se calculó la similitud coseno contra todos los documentos del corpus.

In [ ]:
query = "space exploration and NASA missions"

embedding_query = encoder.encode(query, normalize_embeddings=True)

similitudes = np.dot(embeddings_corpus, embedding_query)

print(f"Similitudes calculadas: {similitudes.shape[0]} documentos")

Similitudes calculadas: 500 documentos


Se obtuvieron los 5 documentos más similares a la query.

In [9]:
top_k = 5
indices_top = np.argsort(similitudes)[::-1][:top_k]

print(f"Query: '{query}'\n")
print(f"Top {top_k} documentos más similares:")
print("-" * 60)

for rank, idx in enumerate(indices_top, 1):
    categoria = nombres_categorias[labels[idx]]
    score = similitudes[idx]
    preview = docs[idx].strip().replace('\n', ' ')[:150]
    print(f"\n{rank}. [Score: {score:.4f}] Categoría: {categoria}")
    print(f"   {preview}...")

Query: 'space exploration and NASA missions'

Top 5 documentos más similares:
------------------------------------------------------------

1. [Score: 0.4026] Categoría: sci.space
   News-Software: UReply 3.1 X-X-From: Wingert@VNET.IBM.com (Bret Wingert)             <C5uBn5.tz@zoo.toronto.edu>   ====================================...

2. [Score: 0.3988] Categoría: sci.space
   In fact, you probably want to avoid US Government anything for such a project.  The pricetag is invariably too high, either in money or in hassles.  T...

3. [Score: 0.3510] Categoría: sci.med
   Newsgroups: sci.med    Path: news.larc.nasa.gov!saimiri.primate.wisc.edu!sdd.hp.com!elroy.jpl.nasa.gov!swrinde!zaphod.mps.ohio-state.edu!howland.resto...

4. [Score: 0.3426] Categoría: sci.space
   All of this talk about a COMMERCIAL space race (i.e. $1G to the first 1-year  moon base) is intriguing. Similar prizes have influenced aerospace  deve...

5. [Score: 0.3418] Categoría: sci.space
   Archive-name: space/schedul

### 2.4 Retrieval-Augmented Generation (RAG)

Se tomaron los documentos recuperados en el paso anterior y se armó un prompt que los incluye como contexto. De esta forma Gemini puede responder basándose en información real del corpus en lugar de solo su conocimiento interno.

In [ ]:
# Se arma el contexto con los top-k documentos recuperados
contexto_docs = ""
for i, idx in enumerate(indices_top, 1):
    texto = docs[idx].strip().replace('\n', ' ')[:500]
    contexto_docs += f"Documento {i} [{nombres_categorias[labels[idx]]}]:\n{texto}\n\n"

prompt = f"""Basándote únicamente en los siguientes documentos recuperados, responde la pregunta del usuario.
Si la información no es suficiente, indícalo.

--- DOCUMENTOS ---
{contexto_docs}
--- PREGUNTA ---
{query}
"""


Se envió el prompt con el contexto a Gemini para obtener una respuesta fundamentada en los documentos recuperados.

In [12]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

display(response.text)

'Los documentos indican que las operaciones del transbordador espacial (Space Shuttle) de NASA, incluyendo sus lanzamientos y aterrizajes, tienen horarios disponibles y se discuten en grupos de Usenet. El incidente del Challenger, relacionado con el Space Shuttle, se menciona en el contexto de procesos operativos.\nEn cuanto a la exploración espacial en general, se discuten los costos y desafíos de llegar a la Luna (especialmente la órbita baja y el aterrizaje), así como la idea de una carrera espacial comercial para establecer una base lunar.\n\nLa información detallada sobre otras misiones específicas de NASA es limitada.'

### 2.5 Consulta con historial de conversación

Se utilizó un `deque` para mantener el historial de la conversación. Se agregaron el prompt original, la respuesta de Gemini y una pregunta de seguimiento, de modo que el modelo tenga contexto de lo que se habló antes.

In [14]:
from collections import deque

contexto = deque()
contexto.append(prompt)
contexto.append(response.text)
contexto.append("¿Cuál de estos documentos menciona más detalles sobre lanzamientos de transbordadores?")

print(f"Historial de conversación: {len(contexto)} mensajes")

Historial de conversación: 3 mensajes


Se envió el historial completo a Gemini para que responda la pregunta de seguimiento tomando en cuenta toda la conversación previa.

In [15]:
response2 = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=list(contexto)
)

print(response2.text)

El Documento 5 menciona más detalles sobre los lanzamientos de transbordadores, indicando que hay horarios disponibles, cobertura televisiva, y un manifiesto con fechas de lanzamiento y otra información, accesible en el grupo de Usenet sci.space.shuttle y el archivo SPACE de Ames.
